# 🔄 Notebook 02 — Transformação & Modelagem Financeira

**Projeto:** Análise de Custos e Margem por Categoria — Olist E-Commerce  
**Autor:** Ariel Marquezin  
**Stack:** DuckDB · pandas · numpy  

---

## 🎯 Objetivo deste notebook

Construir a **camada analítica (Silver/Gold)** do projeto:

1. Ler os Parquet gerados no notebook 01 via **DuckDB** (SQL direto em arquivos)
2. Realizar joins e agregações com queries SQL puras
3. Calcular métricas financeiras: CPV simulado, Receita Líquida, Margem Bruta
4. Montar uma **DRE simplificada por categoria** com **pandas + numpy**
5. Exportar tabela Gold para o notebook de visualização

In [ ]:
# ── Instalação (somente se necessário) ───────────────────────────────────────
# !pip install duckdb pandas numpy -q

In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import warnings
warnings.filterwarnings('ignore')

import duckdb
import pandas as pd
import numpy as np

print(f"DuckDB  : {duckdb.__version__}")
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print("Imports OK ✅")

In [ ]:
# ── 2. Configuração de paths ──────────────────────────────────────────────────
PARQUET_PATH = "../data/parquet/"
GOLD_PATH    = "../data/gold/"
os.makedirs(GOLD_PATH, exist_ok=True)

# Parâmetros do modelo financeiro (configuráveis)
# Em produção, viriam de uma tabela de parâmetros ou config YAML
PARAMS = {
    "cogs_pct"         : 0.55,   # CPV como % do preço (55% — referência e-commerce BR)
    "op_cost_per_day"  : 2.50,   # Custo operacional por dia de entrega (R$)
    "cancel_cost_pct"  : 0.08,   # % do valor do pedido perdido em cancelamentos
    "tax_rate"         : 0.12,   # Impostos sobre receita (simplificado)
}
print("Parâmetros financeiros configurados:")
for k, v in PARAMS.items():
    print(f"  {k:<25} = {v}")

In [ ]:
# ── 3. Conexão DuckDB e registro dos Parquet como views ───────────────────────
# DuckDB lê Parquet diretamente — sem precisar carregar tudo na memória.
# Isso demonstra uso analítico de SQL sem banco de dados.

con = duckdb.connect()  # in-memory

parquet_files = [
    "orders", "order_items", "products",
    "payments", "reviews", "customers", "sellers"
]

for name in parquet_files:
    path = os.path.join(PARQUET_PATH, f"{name}.parquet")
    con.execute(f"""
        CREATE OR REPLACE VIEW {name}
        AS SELECT * FROM read_parquet('{path}')
    """)
    count = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
    print(f"  View criada: {name:<20} ({count:>9,} registros)")

print("\nDuckDB views registradas ✅")

In [ ]:
# ── 4. Join principal — tabela base analítica ─────────────────────────────────
# Une pedidos + itens + produtos + pagamentos + reviews.
# SQL puro no DuckDB — legível e rastreável.

SQL_BASE = """
SELECT
    o.order_id,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp                          AS purchase_date,
    o.order_delivered_customer_date                     AS delivered_date,

    -- Datas de entrega
    DATE_DIFF(
        'day',
        o.order_purchase_timestamp,
        o.order_delivered_customer_date
    )                                                   AS delivery_days,

    -- Produto
    i.product_id,
    COALESCE(p.category, 'unknown')                     AS category,
    i.price,
    i.freight_value,
    i.price + i.freight_value                           AS gross_revenue,

    -- Pagamento
    pay.payment_type,
    pay.payment_installments,
    pay.payment_value,

    -- Avaliação
    COALESCE(r.review_score, 3)                         AS review_score

FROM orders o
JOIN order_items  i   ON o.order_id    = i.order_id
LEFT JOIN products p  ON i.product_id  = p.product_id
LEFT JOIN payments pay ON o.order_id   = pay.order_id
LEFT JOIN reviews  r  ON o.order_id    = r.order_id
WHERE o.order_status NOT IN ('unavailable', 'created')
"""

df_base = con.execute(SQL_BASE).df()
print(f"Tabela base: {len(df_base):,} linhas × {len(df_base.columns)} colunas")
df_base.head(3)

In [ ]:
# ── 5. Cálculo de métricas financeiras com numpy ──────────────────────────────
# numpy é usado para operações vetorizadas — mais rápido que apply() no pandas.

df = df_base.copy()

# Flags
df["is_cancelled"] = np.where(df["order_status"] == "canceled", 1, 0)
df["is_delivered"] = np.where(df["order_status"] == "delivered", 1, 0)

# Receita líquida (desconta cancelamentos e impostos)
df["cancel_loss"]      = df["is_cancelled"] * df["price"] * PARAMS["cancel_cost_pct"]
df["tax"]              = df["price"] * PARAMS["tax_rate"]
df["net_revenue"]      = np.maximum(df["price"] - df["cancel_loss"] - df["tax"], 0)

# CPV simulado = % do preço + frete (custo logístico real da Olist)
df["cogs"]             = df["price"] * PARAMS["cogs_pct"] + df["freight_value"]

# Custo operacional por tempo de entrega
df["delivery_days_c"]  = np.clip(df["delivery_days"].fillna(15), 1, 60)  # cap 60 dias
df["op_cost"]          = df["delivery_days_c"] * PARAMS["op_cost_per_day"]

# Margem bruta e EBITDA estimado
df["gross_profit"]     = df["net_revenue"] - df["cogs"]
df["ebitda"]           = df["gross_profit"] - df["op_cost"]
df["gross_margin_pct"] = np.where(
    df["net_revenue"] > 0,
    (df["gross_profit"] / df["net_revenue"]) * 100,
    np.nan
)

print("Métricas calculadas:")
print(df[["price","net_revenue","cogs","gross_profit","ebitda","gross_margin_pct"]]
      .describe().round(2))

In [ ]:
# ── 6. DRE simplificada por categoria (DuckDB sobre pandas DF) ────────────────
# Registramos o pandas DF como view DuckDB para rodar SQL analítico.

con.register("df_metrics", df)

SQL_DRE = """
SELECT
    category,

    -- Volume
    COUNT(DISTINCT order_id)                            AS total_orders,
    SUM(is_cancelled)                                   AS cancelled_orders,
    ROUND(SUM(is_cancelled)*100.0 / COUNT(*), 1)        AS cancel_rate_pct,

    -- DRE
    ROUND(SUM(price),           2)                      AS receita_bruta,
    ROUND(SUM(cancel_loss),     2)                      AS deducoes_cancelamentos,
    ROUND(SUM(tax),             2)                      AS impostos,
    ROUND(SUM(net_revenue),     2)                      AS receita_liquida,
    ROUND(SUM(cogs),            2)                      AS cpv,
    ROUND(SUM(gross_profit),    2)                      AS lucro_bruto,
    ROUND(SUM(op_cost),         2)                      AS custo_operacional,
    ROUND(SUM(ebitda),          2)                      AS ebitda,

    -- Margens
    ROUND(AVG(gross_margin_pct), 1)                     AS margem_bruta_media_pct,
    ROUND(SUM(ebitda)*100.0 / NULLIF(SUM(net_revenue),0), 1) AS margem_ebitda_pct,

    -- Ticket e frete
    ROUND(AVG(price),           2)                      AS ticket_medio,
    ROUND(AVG(freight_value),   2)                      AS frete_medio,
    ROUND(AVG(review_score),    2)                      AS nps_medio,
    ROUND(AVG(delivery_days_c), 1)                      AS prazo_entrega_medio_dias

FROM df_metrics
GROUP BY category
HAVING total_orders >= 50     -- filtra categorias com volume mínimo
ORDER BY receita_liquida DESC
"""

df_dre = con.execute(SQL_DRE).df()
print(f"DRE por categoria: {len(df_dre)} categorias")
df_dre.head(10)

In [ ]:
# ── 7. Rankings e insights com pandas ────────────────────────────────────────

df_dre["rank_receita"]  = df_dre["receita_liquida"].rank(ascending=False).astype(int)
df_dre["rank_margem"]   = df_dre["margem_bruta_media_pct"].rank(ascending=False).astype(int)
df_dre["rank_ebitda"]   = df_dre["ebitda"].rank(ascending=False).astype(int)

# Quadrante: Alto Volume × Alta Margem
receita_median = df_dre["receita_liquida"].median()
margem_median  = df_dre["margem_bruta_media_pct"].median()

def quadrant(row):
    high_rev = row["receita_liquida"]       >= receita_median
    high_mar = row["margem_bruta_media_pct"] >= margem_median
    if high_rev and high_mar:     return "⭐ Alto Volume / Alta Margem"
    elif high_rev and not high_mar: return "📦 Alto Volume / Baixa Margem"
    elif not high_rev and high_mar: return "💎 Nicho Rentável"
    else:                           return "⚠️  Baixo Volume / Baixa Margem"

df_dre["quadrant"] = df_dre.apply(quadrant, axis=1)

print("\nDistribuição por quadrante:")
print(df_dre["quadrant"].value_counts().to_string())

print("\nTop 5 por EBITDA:")
print(df_dre.nlargest(5, "ebitda")[["category","ebitda","margem_ebitda_pct","quadrant"]].to_string(index=False))

print("\nBottom 5 por EBITDA:")
print(df_dre.nsmallest(5, "ebitda")[["category","ebitda","margem_ebitda_pct","cancel_rate_pct"]].to_string(index=False))

In [ ]:
# ── 8. Window Functions via DuckDB — Participação Acumulada (Pareto) ──────────

SQL_PARETO = """
WITH ranked AS (
    SELECT
        category,
        receita_liquida,
        SUM(receita_liquida) OVER ()                        AS total_receita,
        receita_liquida / SUM(receita_liquida) OVER () * 100 AS participacao_pct,
        SUM(receita_liquida) OVER (
            ORDER BY receita_liquida DESC
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) / SUM(receita_liquida) OVER () * 100              AS participacao_acumulada_pct,
        ROW_NUMBER() OVER (ORDER BY receita_liquida DESC)   AS rank
    FROM df_dre
)
SELECT * FROM ranked ORDER BY rank
"""

df_pareto = con.execute(SQL_PARETO).df()

# Quantas categorias respondem por 80% da receita?
top80 = df_pareto[df_pareto["participacao_acumulada_pct"] <= 80]
print(f"Princípio de Pareto:")
print(f"  {len(top80)} categorias respondem por 80% da receita líquida total")
print(f"  de {len(df_pareto)} categorias com volume >= 50 pedidos")
df_pareto.head(10)

In [ ]:
# ── 9. Exportar tabelas Gold ──────────────────────────────────────────────────
import pyarrow as pa
import pyarrow.parquet as pq

def save_gold(df_out: pd.DataFrame, name: str):
    path  = os.path.join(GOLD_PATH, f"{name}.parquet")
    table = pa.Table.from_pandas(df_out, preserve_index=False)
    pq.write_table(table, path, compression="snappy")
    print(f"  Gold salvo: {name}.parquet  ({len(df_out):,} linhas)")

save_gold(df_dre,    "dre_por_categoria")
save_gold(df_pareto, "pareto_receita")
save_gold(df,        "metricas_pedidos")

print("\nCamada Gold exportada ✅")

## ✅ Resumo — Notebook 02

| Etapa | Ferramenta | Resultado |
|-------|------------|----------|
| Leitura Parquet | DuckDB | Views SQL sobre arquivos sem carregar memória |
| Join analítico | DuckDB SQL | Tabela base com 100k+ registros |
| Métricas financeiras | numpy (vetorizado) | CPV, Receita Líquida, Margem, EBITDA |
| DRE por categoria | DuckDB + pandas | 40+ categorias com P&L completo |
| Análise de Pareto | DuckDB Window Functions | Top X% categorias = 80% da receita |
| Exportação Gold | PyArrow Parquet | 3 tabelas prontas para visualização |

**Próximo passo →** `03_analise_cpv.ipynb` — queries SQL avançadas e análise de giro de estoque.  
**Depois →** `04_visualizacao.ipynb` — Matplotlib + Seaborn com storytelling STAR.